In [ ]:
# ============================================================
# BLOCK 1: PREPROCESSING
# ============================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import LocalOutlierFactor
from imblearn.over_sampling import SVMSMOTE
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, RFECV, SelectKBest, SelectPercentile, SelectFromModel, VarianceThreshold
from sklearn.metrics import make_scorer, f1_score
from sklearn.feature_selection import mutual_info_classif

columns_to_normalize = [col for col in selected_columns_1 if col != 'participantid']

def preprocess_fold(X_train, X_test, y_train):
    # RobustScaler per platform (fitted on training fold only)
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train[columns_to_normalize] = X_train[columns_to_normalize].astype(float)
    X_test[columns_to_normalize] = X_test[columns_to_normalize].astype(float)
    for p in [1, 0]:
        tr = X_train['platform'] == p
        te = X_test['platform'] == p
        scaler = RobustScaler()
        X_train.loc[tr, columns_to_normalize] = scaler.fit_transform(X_train.loc[tr, columns_to_normalize])
        if te.any():
            X_test.loc[te, columns_to_normalize] = scaler.transform(X_test.loc[te, columns_to_normalize])
    X_train = X_train.drop(columns='platform')
    X_test = X_test.drop(columns='platform')

    # LOF
    X = X_train
    y = y_train
    outliers_sets = []
    neighbors_list = [5, 10, 15, 20, 25, 30]
    for n_neighbors in neighbors_list:
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=0.05)
        preds = lof.fit_predict(X)
        outliers_idx = np.where(preds == -1)[0]
        outliers_sets.append(set(outliers_idx))
        print(f"n_neighbors={n_neighbors}: Detected {len(outliers_idx)} outliers")
    consistent_outliers = set.intersection(*outliers_sets)
    print(f"Consistently identified outliers across all runs: {len(consistent_outliers)}")
    mask = np.ones(len(X), dtype=bool)
    mask[list(consistent_outliers)] = False
    X_clean = X[mask]
    y_clean = y[mask]
    X_final_cleaned = X_clean
    y_final_cleaned = y_clean

    # SVMSMOTE
    X = X_final_cleaned
    y = y_final_cleaned
    oversample = SVMSMOTE( sampling_strategy='auto',  random_state=None, k_neighbors=2, m_neighbors=4, svm_estimator=None, out_step=0.5)
    X_resampled, y_resampled= oversample.fit_resample(X, y)
    class_counts = pd.Series(y_resampled).value_counts()
    print(class_counts)
    X = X_resampled
    y = y_resampled

    # Feature selection
    scorer = make_scorer(f1_score, average='macro')
    Features_Name = X.columns
    feature_selection_methods = [
        "RFECV", "SelectFromModel", "MutualInfo", "RFE",
        "VarianceThreshold", "SelectKBest", "SelectPercentile", "MutualInfo_classif"
    ]
    results = {}
    rf_param_grid = {
        'n_estimators': [10, 20, 30, 40, 50],
        'max_depth': [2, 3, 4, 5],
        'min_samples_split': [2, 3, 4, 5, 6],
        'min_samples_leaf': [2, 3, 4],
        'max_features': ['sqrt', 'log2'],
    }
    for method in feature_selection_methods:
        best_score = -np.inf
        best_cv_value = None
        best_n_features = None
        best_features = None
        best_feature_names = None
        for cv_value in range(2, 11):
            cv = StratifiedKFold(n_splits=cv_value)
            if method == "RFECV":
                selector = RFECV(estimator=RandomForestClassifier(random_state=42), step=1, cv=cv, scoring=scorer, n_jobs=-1)
            elif method == "SelectFromModel":
                selector = SelectFromModel(estimator=RandomForestClassifier(random_state=42))
            elif method in ["MutualInfo", "SelectKBest", "MutualInfo_classif"]:
                selector = SelectKBest(mutual_info_classif, k='all')
            elif method == "RFE":
                selector = RFE(estimator=RandomForestClassifier(random_state=42), n_features_to_select=5)
            elif method == "SelectPercentile":
                selector = SelectPercentile(mutual_info_classif, percentile=50)
            elif method == "VarianceThreshold":
                selector = VarianceThreshold()
            selector.fit(X, y)
            X_selected = selector.transform(X) if hasattr(selector, 'transform') else X
            grid_search = GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=cv, scoring=scorer, n_jobs=-1)
            grid_search.fit(X_selected, y)
            mean_score = grid_search.best_score_
            if mean_score > best_score:
                best_score = mean_score
                best_cv_value = cv_value
                best_n_features = X_selected.shape[1]
                best_features = selector.get_support() if hasattr(selector, 'get_support') else None
                best_feature_names = Features_Name[best_features] if best_features is not None else None
        results[method] = {
            'Best CV Value': best_cv_value,
            'Best Number of Features': best_n_features,
            'Best Score': best_score,
            'Best Features': best_feature_names,
            'Best RF Params': grid_search.best_params_
        }
    df_results = pd.DataFrame(results).T
    df_results = df_results.sort_values(by="Best Score", ascending=False)

    best_method = df_results.index[0]
    best_cv_value = df_results.loc[best_method, "Best CV Value"]
    best_features = df_results.loc[best_method, "Best Features"]
    best_rf_params = df_results.loc[best_method, "Best RF Params"]

    X_new = X[best_features] if best_features is not None else X
    X_test_new = X_test[best_features] if best_features is not None else X_test

    print(f"\nBest Method: {best_method}")
    print(f"Best CV Value: {best_cv_value}")
    print(f"Selected Features:\n{best_features}")
    print(f"Best RandomForest Parameters:\n{best_rf_params}")

    return X_new, y, X_test_new, int(best_cv_value)


# ============================================================
# BLOCK 2: CLASSIFIERS (keep one uncommented per run)
# ============================================================
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# CatBoostClassifier
classifiers = [
    CatBoostClassifier(verbose=0),
]
param_grids = [
    {
        'iterations': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110],
        'learning_rate': [0.01, 0.05, 0.1],
        'depth': [1, 2, 3, 4, 5],
    },
]

# # XGBClassifier
# classifiers = [
#     XGBClassifier(),
# ]
# param_grids = [
#     {
#         'n_estimators': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110],
#         'learning_rate': [0.01, 0.05, 0.1],
#         'max_depth': [1, 2, 3, 4, 5],
#     },
# ]

# # RandomForestClassifier
# classifiers = [
#     RandomForestClassifier(),
# ]
# param_grids = [
#     {
#         'n_estimators': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110],
#         'max_depth': [1, 2, 3, 4, 5],
#         'min_samples_split': [2, 3, 4],
#         'min_samples_leaf': [1, 2, 3],
#         'max_features': ['sqrt', 'log2'],
#     },
# ]

# # GradientBoostingClassifier
# classifiers = [
#     GradientBoostingClassifier(),
# ]
# param_grids = [
#     {
#         'n_estimators': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110],
#         'learning_rate': [0.01, 0.05, 0.1],
#         'max_depth': [1, 2, 3, 4, 5],
#     },
# ]

# # ExtraTreesClassifier
# classifiers = [
#     ExtraTreesClassifier(),
# ]
# param_grids = [
#     {
#         'n_estimators': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110],
#         'max_depth': [1, 2, 3, 4, 5],
#         'min_samples_split': [2, 3, 4],
#         'min_samples_leaf': [1, 2, 3],
#         'max_features': ['sqrt', 'log2'],
#     },
# ]


# ============================================================
# BLOCK 3: NESTED CV
# ============================================================
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             balanced_accuracy_score, recall_score,
                             confusion_matrix, roc_curve)
import matplotlib.pyplot as plt
import numpy as np

# Outer Cross Validation
best_cv_value = 10
outer_cv = StratifiedKFold(n_splits=best_cv_value, shuffle=True, random_state=42)

# Initialize lists and dictionaries
outer_results = []
best_models_per_fold = {}
inner_best_models = {}

for fold_idx, (train_index, test_index) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n========= Fold {fold_idx} =========")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Preprocessing inside the fold
    X_train, y_train, X_test, fold_cv_value = preprocess_fold(X_train, X_test, y_train)

    # Inner CV
    inner_cv = StratifiedKFold(n_splits=fold_cv_value, shuffle=True, random_state=42)

    best_classifier = None
    best_params = None
    best_score = 0.0
    best_model_name = None

    for clf, params in zip(classifiers, param_grids):
        model_name = clf.__class__.__name__
        print(f"Evaluating {model_name}...")

        scoring_metrics = {'accuracy': 'accuracy', 'f1_weighted': 'f1_weighted'}
        grid_search = GridSearchCV(estimator=clf, param_grid=params, cv=inner_cv,
                                   scoring=scoring_metrics, refit='accuracy')
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        y_pred_inner = best_model.predict(X_test)
        acc = accuracy_score(y_test, y_pred_inner)
        f1 = f1_score(y_test, y_pred_inner, average='weighted')

        inner_best_models[(fold_idx, model_name)] = {
            'model': best_model,
            'params': grid_search.best_params_,
            'f1_score': f1,
            'accuracy': acc
        }

        if acc > best_score:
            best_score = acc
            best_classifier = best_model
            best_model_name = model_name
            best_params = grid_search.best_params_

    if best_classifier is None:
        best_classifier = classifiers[0]
        best_model_name = best_classifier.__class__.__name__
        print("No best model found, using default.")

    print(f"Best Hyperparameters from Inner CV for Fold {fold_idx}: "
          f"{best_model_name} with {best_params}")

    # Outer test evaluation
    y_pred_outer = best_classifier.predict(X_test)
    y_prob_outer = best_classifier.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred_outer)
    f1 = f1_score(y_test, y_pred_outer, average='weighted')
    auc_score = roc_auc_score(y_test, y_prob_outer)
    bal_acc = balanced_accuracy_score(y_test, y_pred_outer)
    sens = recall_score(y_test, y_pred_outer, pos_label=1)

    cm = confusion_matrix(y_test, y_pred_outer)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    spec = tn / (tn + fp) if (tn + fp) else 0.0

    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc_score:.4f}, "
          f"Balanced Acc: {bal_acc:.4f}, Sensitivity: {sens:.4f}, Specificity: {spec:.4f}")

    outer_results.append({
        'fold': fold_idx,
        'model_name': best_model_name,
        'accuracy': acc,
        'f1_score': f1,
        'auc': auc_score,
        'balanced_accuracy': bal_acc,
        'sensitivity': sens,
        'specificity': spec
    })

    best_models_per_fold[fold_idx] = {
        'model': best_classifier,
        'model_name': best_model_name,
        'params': best_params,
        'metrics': outer_results[-1]
    }

    # ROC Curve for the fold
    fpr, tpr, _ = roc_curve(y_test, y_prob_outer)
    plt.plot(fpr, tpr, label=f'Fold {fold_idx} - {best_model_name} (AUC={auc_score:.2f})')

# Final ROC Plot
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Best Model in Each Fold')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Summary
print("\n======= Final Summary =======")
for res in outer_results:
    print(f"Fold {res['fold']} - {res['model_name']}: "
          f"Acc={res['accuracy']:.4f}, F1={res['f1_score']:.4f}, AUC={res['auc']:.4f}, "
          f"BalAcc={res['balanced_accuracy']:.4f}, Sens={res['sensitivity']:.4f}, Spec={res['specificity']:.4f}")

# Mean ± Std Summary
def summarize_metric(metric):
    values = [res[metric] for res in outer_results]
    return f"{np.mean(values):.4f} ± {np.std(values):.4f}"

print(f"\nOverall Accuracy: {summarize_metric('accuracy')}")
print(f"Overall F1 Score: {summarize_metric('f1_score')}")
print(f"Overall AUC: {summarize_metric('auc')}")
print(f"Overall Balanced Accuracy: {summarize_metric('balanced_accuracy')}")
print(f"Overall Sensitivity: {summarize_metric('sensitivity')}")
print(f"Overall Specificity: {summarize_metric('specificity')}")